# Линейная регрессия — мини-проект

Сквозной мини-проект на реальных данных. Повторяет ключевые темы урока в одном пайплайне.


In [ ]:
# Setup
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.linear_model import HuberRegressor, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

np.random.seed(42)


## Постановка задачи `(intro)`


Предскажем **медианную стоимость жилья** в Калифорнии (OpenML `california_housing`) по демографическим и географическим признакам.

В одном сценарии пройдём: EDA → выбросы → train/test → Pipeline → регуляризация → метрики → диагностика остатков → интерпретация весов.


## Загрузка данных и первичный осмотр `(eda)`


Целевая переменная — `median_house_value` ($). Числовые признаки: доход, возраст домов, комнаты, население, координаты.


In [ ]:
warnings.filterwarnings("ignore")



sns.set_theme(style="whitegrid", font_scale=1.05)

raw = fetch_openml(name="california_housing", version=1, as_frame=True, parser="auto")
df = raw.frame.copy()
target_col = "median_house_value"
feature_cols = [c for c in df.columns if c not in (target_col, "ocean_proximity")]
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

print(f"Объектов: {len(df)}, признаков: {len(feature_cols)}")
display(df.head())
display(df.describe().round(2))


## Разведочный анализ `(viz)`


Сильнейший линейный сигнал — `median_income`. Корреляции подскажут мультиколлинеарность (`total_rooms` ↔ `total_bedrooms`).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df[target_col], bins=40, color="steelblue", edgecolor="white")
axes[0].set_xlabel("median_house_value")
axes[0].set_title("Распределение цены")

axes[1].scatter(df["median_income"], df[target_col], alpha=0.25, s=12, color="steelblue")
axes[1].set_xlabel("median_income")
axes[1].set_ylabel("median_house_value")
axes[1].set_title("Цена vs доход")

corr = df[feature_cols + [target_col]].corr()
sns.heatmap(corr, ax=axes[2], cmap="RdBu_r", center=0, vmin=-1, vmax=1)
axes[2].set_title("Корреляции")
plt.tight_layout()
plt.show()

print("Корр. total_rooms–total_bedrooms:", corr.loc["total_rooms", "total_bedrooms"].round(3))
print("Корр. median_income–цена:", corr.loc["median_income", target_col].round(3))


## Выбросы и рычаг `(experiment)`


MSE сильно тянется к выбросам в **Y**. Точки с экстремальным **X** (leverage) меняют наклон даже при нормальном Y.


In [ ]:
X1 = df[["median_income"]].values
y = df[target_col].values

# выброс в Y
y_out = y.copy()
out_y_idx = int(np.argmax(y))
y_out[out_y_idx] = y.max() * 1.8

ols_clean = LinearRegression().fit(X1, y)
ols_out = LinearRegression().fit(X1, y_out)
huber_out = HuberRegressor().fit(X1, y_out)

x_plot = np.linspace(X1.min(), X1.max(), 200).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X1, y, alpha=0.3, s=10, label="данные")
axes[0].scatter(X1[out_y_idx], y_out[out_y_idx], color="orange", s=80, label="выброс Y", zorder=5)
axes[0].plot(x_plot, ols_clean.predict(x_plot), "g--", label="OLS без выброса")
axes[0].plot(x_plot, ols_out.predict(x_plot), "r-", label="OLS с выбросом")
axes[0].plot(x_plot, huber_out.predict(x_plot), "b:", linewidth=2, label="Huber")
axes[0].set_xlabel("median_income")
axes[0].set_ylabel("median_house_value")
axes[0].set_title("Выброс в целевой переменной")
axes[0].legend(fontsize=8)

# leverage: экстремальный median_income при типичном Y
X_lev = np.vstack([X1, [[X1.max() * 1.15]]])
y_lev = np.append(y, np.median(y))
ols_lev = LinearRegression().fit(X_lev, y_lev)
x_plot2 = np.linspace(X1.min(), X_lev.max(), 200).reshape(-1, 1)
axes[1].scatter(X1, y, alpha=0.3, s=10)
axes[1].scatter(X_lev[-1], y_lev[-1], color="purple", s=80, label="leverage X", zorder=5)
axes[1].plot(x_plot, ols_clean.predict(x_plot), "g--", label="OLS исходные")
axes[1].plot(x_plot2, ols_lev.predict(x_plot2), "r-", label="OLS + leverage")
axes[1].set_xlabel("MedInc")
axes[1].set_title("Выброс по оси X (рычаг)")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


## Train / test и Pipeline без утечки `(example)`


`StandardScaler` обучаем **только на train**. Иначе статистики test «протекают» в обучение.


In [ ]:
X = df[feature_cols].values
y = df[target_col].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe_ols = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
pipe_ols.fit(X_train, y_train)
y_pred = pipe_ols.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"Test MAE:  {mae:,.0f} ($)")
print(f"Test RMSE: {rmse:.3f}")
print(f"Test R2:   {r2:.3f}")


## Регуляризация: Ridge и Lasso `(experiment)`


Ridge (L2) сжимает веса при мультиколлинеарности. Lasso (L1) может обнулять слабые признаки.


In [ ]:
models = {
    "OLS": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "Lasso": Pipeline([("scaler", StandardScaler()), ("model", Lasso(alpha=0.01, max_iter=10000))]),
}

rows = []
coef_rows = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    pred_tr = pipe.predict(X_train)
    pred_te = pipe.predict(X_test)
    reg = pipe.named_steps["model"]
    w = reg.coef_
    rows.append({
        "model": name,
        "R2 train": r2_score(y_train, pred_tr),
        "R2 test": r2_score(y_test, pred_te),
        "RMSE test": mean_squared_error(y_test, pred_te) ** 0.5,
        "norm_w": np.linalg.norm(w),
    })
    for fname, wi in zip(feature_cols, w):
        coef_rows.append({"model": name, "feature": fname, "weight": wi})

metrics_df = pd.DataFrame(rows)
display(metrics_df.round(4))

coef_df = pd.DataFrame(coef_rows)
fig, ax = plt.subplots(figsize=(10, 5))
for i, name in enumerate(models):
    sub = coef_df[coef_df["model"] == name]
    ax.bar(np.arange(len(feature_cols)) + i * 0.25, sub["weight"], width=0.25, label=name)
ax.set_xticks(np.arange(len(feature_cols)) + 0.25)
ax.set_xticklabels(feature_cols, rotation=45, ha="right")
ax.axhline(0, color="k", linewidth=0.5)
ax.set_ylabel("вес (после StandardScaler)")
ax.set_title("Сравнение весов OLS / Ridge / Lasso")
ax.legend()
plt.tight_layout()
plt.show()


## Диагностика остатков `(viz)`


Идеал — случайное облако вокруг нуля. «Воронка» или U-образный паттерн намекают на нелинейность или пропущенные признаки.


In [ ]:
pipe_ols.fit(X_train, y_train)
y_hat = pipe_ols.predict(X_test)
residuals = y_test - y_hat

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(y_hat, residuals, alpha=0.35, s=15, color="steelblue")
axes[0].axhline(0, color="crimson", linestyle="--")
axes[0].set_xlabel("предсказание ŷ")
axes[0].set_ylabel("остаток y − ŷ")
axes[0].set_title("Residual plot")

axes[1].hist(residuals, bins=35, color="steelblue", edgecolor="white")
axes[1].set_xlabel("остаток")
axes[1].set_title("Гистограмма остатков")
plt.tight_layout()
plt.show()


## Интерпретация весов `(viz)`


После масштабирования сравниваем силу признаков. Помним **ceteris paribus**: вес — не причинно-следственный эффект при коррелированных фичах.


In [ ]:
w = pipe_ols.named_steps["model"].coef_
order = np.argsort(np.abs(w))

plt.figure(figsize=(8, 5))
plt.barh(np.array(feature_cols)[order], w[order], color="steelblue")
plt.xlabel("вес (StandardScaler → LinearRegression)")
plt.title("Интерпретация: влияние признаков на median_house_value")
plt.tight_layout()
plt.show()

for name, val in sorted(zip(feature_cols, w), key=lambda t: abs(t[1]), reverse=True)[:4]:
    print(f"{name:12s}: {val:+.3f}")


## Когда линейная модель не хватает `(experiment)`


Зависимость `median_income` → цена слегка нелинейна на хвостах. Полиномиальный признак может улучшить fit — ценой интерпретируемости.


In [ ]:
inc_idx = feature_cols.index("median_income")
poly_pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("poly_inc", PolynomialFeatures(degree=2, include_bias=False), [inc_idx]),
        ("rest", "passthrough", [i for i in range(len(feature_cols)) if i != inc_idx]),
    ])),
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

for name, pipe in [("Linear", pipe_ols), ("Poly income^2", poly_pipe)]:
    pipe.fit(X_train, y_train)
    r2_tr = r2_score(y_train, pipe.predict(X_train))
    r2_te = r2_score(y_test, pipe.predict(X_test))
    print(f"{name:14s} R2 train={r2_tr:.3f}  test={r2_te:.3f}  delta={r2_tr - r2_te:.3f}")


## Чек-лист мини-проекта `(summary)`


1. EDA и scatter до обучения.
2. `train_test_split` до любой трансформации.
3. `Pipeline` + `StandardScaler` для интерпретации и регуляризации.
4. Сравнить OLS / Ridge / Lasso по test RMSE и норме весов.
5. Residual plot — адекватность модели.
6. Интерпретация весов только на масштабированных признаках.
7. Если R2 train >> R2 test — переобучение; если паттерн в остатках — нелинейность.
